# Preprocessing ObRail Europe

Ce notebook prototype toutes les transformations appliquées au dataset brut.
Une fois validées, ces transformations seront industrialisées dans `src/preprocessing.py`.

Entrée : `data/raw/dataset_final.csv`
Sorties :
- `data/processed/dataset_cleaned.csv`
- `data/splits/X_train.csv`, `X_val.csv`, `X_test.csv`
- `data/splits/y_train.csv`, `y_val.csv`, `y_test.csv`
- `data/processed/target_encoding.csv`

In [2]:
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Chemins relatifs au projet
BASE_DIR = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
RAW_DIR       = os.path.join(BASE_DIR, 'data', 'raw')
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
SPLITS_DIR    = os.path.join(BASE_DIR, 'data', 'splits')

# Créer les dossiers s'ils n'existent pas
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(SPLITS_DIR, exist_ok=True)

print("Imports OK")
print(f"BASE_DIR : {BASE_DIR}")

Imports OK
BASE_DIR : c:\Users\josep\Mspr2\MSPR2\MSPR\ml


In [3]:
# Chargement du dataset brut
df = pd.read_csv(os.path.join(RAW_DIR, 'dataset_final.csv'))

print(f"Dataset chargé : {df.shape[0]} lignes, {df.shape[1]} colonnes")
df.head()

Dataset chargé : 400 lignes, 12 colonnes


,distance_km,empreinte_train_kg,empreinte_avion_kg,ratio_co2,operateur,pays_operateur,trajet_id,gare_depart,gare_arrivee,heure_depart,heure_arrivee,type_service
0,1115.0,19.2,176.170,0.1090,Укрзалізниця,UA,UZ 064,Lviv,Kharkiv,1899-12-30T15:30:00.000Z,1899-12-30T05:56:00.000Z,NUIT
1,875.0,2.6,138.250,0.0188,SNCF Voyageurs SA,FR,SNCF IC Nuit 3971,Paris Austerlitz,Latour-de-Carol / La Tor de Querol,1899-12-30T21:40:00.000Z,1899-12-30T10:07:00.000Z,NUIT
2,1032.0,20.8,163.056,0.1276,Trenitalia S.p.A,IT,FS IC Notte 755,Milano Centrale,Lecce,1899-12-30T21:50:00.000Z,1899-12-30T09:30:00.000Z,NUIT
3,867.0,17.7,136.986,0.1292,Trenitalia S.p.A,IT,FS IC Notte 1954,Palermo Centrale,Roma Termini,1899-12-30T18:48:00.000Z,1899-12-30T07:18:00.000Z,NUIT
4,493.0,10.3,77.894,0.1322,C.F.R. Călători S.A.,RO,CFR 1641 (CN),București Nord,Cluj-Napoca,1899-12-30T21:03:00.000Z,1899-12-30T07:02:00.000Z,NUIT


### Nettoyage : suppression des colonnes non pertinentes

**trajet_id** est supprimé car c'est un identifiant unique sans valeur prédictive.

**gare_depart** et **gare_arrivee** sont supprimées car elles présentent trop 
de modalités distinctes et leur information géographique est déjà capturée 
par **distance_km**. Principe de minimisation des données RGPD Art. 5.1.c.

**pays_operateur** est supprimé car cette information est déjà portée 
par **operateur** qui sera encodé par ses émissions moyennes. 
Garder les deux introduirait une redondance.

In [4]:
# Suppression des colonnes exclues du modèle
colonnes_a_supprimer = ['trajet_id', 'gare_depart', 'gare_arrivee', 'pays_operateur']
df = df.drop(columns=colonnes_a_supprimer)

print(f"Colonnes restantes : {df.shape[1]}")
print(df.columns.tolist())

Colonnes restantes : 8
['distance_km', 'empreinte_train_kg', 'empreinte_avion_kg', 'ratio_co2', 'operateur', 'heure_depart', 'heure_arrivee', 'type_service']


### Feature engineering : durée du trajet

**heure_depart** et **heure_arrivee** sont des chaînes de caractères 
sans valeur directe pour le modèle. On les transforme en **duree_trajet_min** 
qui représente la durée réelle du trajet en minutes.

Les trains de nuit partent le soir et arrivent le lendemain matin. 
La soustraction brute donnerait une durée négative dans ce cas. 
On corrige en ajoutant 1440 minutes (24h) aux durées négatives.

Une fois **duree_trajet_min** calculée, **heure_depart** et **heure_arrivee** 
sont supprimées.

In [5]:
df['heure_depart'] = pd.to_datetime(df['heure_depart'])
df['heure_arrivee'] = pd.to_datetime(df['heure_arrivee'])

df['duree_trajet_min'] = (df['heure_arrivee'] - df['heure_depart']).dt.total_seconds() / 60

df['duree_trajet_min'] = df['duree_trajet_min'].apply(
    lambda x: x + 1440 if x < 0 else x
)

df = df.drop(columns=['heure_depart', 'heure_arrivee'])

print(f"Durée min : {df['duree_trajet_min'].min():.0f} min")
print(f"Durée max : {df['duree_trajet_min'].max():.0f} min")
print(df[['distance_km', 'duree_trajet_min']].head(10))

Durée min : 8 min
Durée max : 1362 min
   distance_km  duree_trajet_min
0       1115.0             866.0
1        875.0             747.0
2       1032.0             700.0
3        867.0             750.0
4        493.0             599.0
5        849.0            1041.0
6        561.0             283.0
7        760.0             696.0
8       1209.0            1167.0
9       1110.0            1208.0


In [6]:
print(df[df['duree_trajet_min'] < 30][['distance_km', 'duree_trajet_min', 'operateur']].sort_values('duree_trajet_min'))

     distance_km  duree_trajet_min     operateur
283       1212.0               8.0  Укрзалізниця
42        1036.0              17.0  Укрзалізниця
206       1036.0              27.0  Укрзалізниця


### Anomalies détectées sur duree_trajet_min

Trois trajets présentent des durées inférieures à 60 minutes 
pour des distances supérieures à 1000 km, ce qui est physiquement impossible.

Ces anomalies proviennent d'erreurs dans les heures de départ et d'arrivée 
enregistrées en base de données pour ces trajets Укрзалізниця.

Ces 3 lignes représentent 0.75% du dataset. Leur suppression est préférable 
à une imputation qui introduirait des valeurs artificielles. 
Le dataset passera de 400 à 397 lignes après suppression.

In [8]:
# Suppression des trajets avec durée < 30 min ET distance > 1000 km
avant = df.shape[0]
df = df[~((df['duree_trajet_min'] < 30) & (df['distance_km'] > 1000))].reset_index(drop=True)
apres = df.shape[0]

print(f"Lignes supprimées : {avant - apres}")
print(f"Dataset final : {apres} lignes")
print(f"Durée min après nettoyage : {df['duree_trajet_min'].min():.0f} min")
print(f"Durée max : {df['duree_trajet_min'].max():.0f} min")

Lignes supprimées : 0
Dataset final : 393 lignes
Durée min après nettoyage : 86 min
Durée max : 1362 min


### Résultat du nettoyage

7 lignes ont été supprimées car elles présentaient des durées inférieures 
à 30 minutes, sont incohérentes avec les distances enregistrées 
pour ces trajets. Ces anomalies semblent liées à des erreurs dans les heures 
de départ ou d'arrivée en base de données.

Le dataset passe de 400 à 393 lignes. La durée minimale est désormais 
de 86 minutes.

### Encodage des variables catégorielles

Deux variables catégorielles doivent être transformées en valeurs numériques 
avant l'entraînement des modèles.

**operateur** sera encodé par Target Encoding : chaque opérateur est remplacé 
par la moyenne de **empreinte_train_kg** calculée uniquement sur les données 
d'entraînement. Cette approche est robuste au déséquilibre des opérateurs 
et évite la malédiction de la dimensionnalité qu'introduirait un One-Hot Encoding 
sur 25 modalités.

**type_service** sera encodé en binaire : JOUR = 0, NUIT = 1.

Note : le Target Encoding sera fitté uniquement sur le train après le split 
pour éviter toute fuite de données vers la validation et le test.

In [9]:
# Encodage binaire type_service
df['type_service'] = df['type_service'].map({'JOUR': 0, 'NUIT': 1})

print("Encodage type_service :")
print(df['type_service'].value_counts())

Encodage type_service :
type_service
1    303
0     90
Name: count, dtype: int64


### Split des données

Le dataset est découpé en trois ensembles avant d'appliquer le Target Encoding 
sur operateur, afin d'éviter tout data leakage.

70% entraînement, 15% validation, 15% test avec random_state=42.

Le train sert à apprendre. La validation sert à comparer les modèles 
et choisir le meilleur sans toucher au test. Le test sert à l'évaluation 
finale une seule fois, pour avoir un score honnête qui n'a pas été 
influencé par les choix de modélisation.

In [10]:
from sklearn.model_selection import train_test_split

# Features et cible pour la régression
features_regression = ['distance_km', 'operateur', 'type_service', 'duree_trajet_min']
cible = 'empreinte_train_kg'

X = df[features_regression]
y = df[cible]

# Split 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print(f"Train : {X_train.shape[0]} lignes")
print(f"Validation : {X_val.shape[0]} lignes")
print(f"Test : {X_test.shape[0]} lignes")

Train : 275 lignes
Validation : 59 lignes
Test : 59 lignes


### Target Encoding sur operateur

Le Target Encoding est appliqué après le split pour éviter le data leakage.
Si on l'avait appliqué avant, les moyennes auraient été calculées 
sur l'ensemble du dataset, y compris les données de validation et de test. 
Le modèle aurait indirectement vu ces données pendant l'entraînement.

Concrètement, on calcule la moyenne de **empreinte_train_kg** 
par opérateur uniquement sur les 275 lignes d'entraînement.
Cette moyenne remplace le nom de l'opérateur dans les trois ensembles.

PKP Intercity qui émet en moyenne 23.86 kg devient 23.86.
SNCF qui émet en moyenne 3.02 kg devient 3.02.

Si un opérateur apparaît dans la validation ou le test 
mais pas dans le train, il reçoit la moyenne globale du train 
comme valeur par défaut.

In [12]:
# Target Encoding sur operateur
# Calculé uniquement sur le train pour éviter le data leakage
target_encoding = X_train.copy()
target_encoding['empreinte_train_kg'] = y_train.values

encoding_map = target_encoding.groupby('operateur')['empreinte_train_kg'].mean()

# Appliquer sur train, val et test
X_train['operateur'] = X_train['operateur'].map(encoding_map)
X_val['operateur']   = X_val['operateur'].map(encoding_map)
X_test['operateur']  = X_test['operateur'].map(encoding_map)

# Gérer les opérateurs non vus dans le train (valeur par défaut = moyenne globale)
moyenne_globale = y_train.mean()
X_train['operateur'] = X_train['operateur'].fillna(moyenne_globale)
X_val['operateur']   = X_val['operateur'].fillna(moyenne_globale)
X_test['operateur']  = X_test['operateur'].fillna(moyenne_globale)

# Sauvegarder la table d'encodage
encoding_map.reset_index().rename(
    columns={'empreinte_train_kg': 'target_encoding'}
).to_csv(os.path.join(PROCESSED_DIR, 'target_encoding.csv'), index=False)

print("Target Encoding appliqué :")
print(encoding_map.sort_values(ascending=False).round(2))

Target Encoding appliqué :
operateur
PKP Intercity S.A.                       23.10
MÁV-START Vasúti Személyszállító Zrt.    22.08
TCDD Taşımacılık A.Ş.                    20.43
Trenitalia S.p.A                         18.78
České dráhy a.s.                         17.58
Укрзалізниця                             17.25
RegioJet a.s.                            15.90
Caledonian Sleeper Ltd.                  15.77
Astra Trans Carpatic SRL                 15.67
C.F.R. Călători S.A.                     15.04
Железнице Србије ад                      14.68
ÖBB-Personenverkehr AG                   14.18
First Greater Western Ltd.               13.40
European Sleeper Cooperatïe              13.39
VR-Yhtymä Oy                             12.78
БДЖ - Пътнически превози ЕООД            12.25
Calea Ferată din Moldova                 11.68
Merresor AB                              11.10
Železničná spoločnosť Slovensko a.s.      8.36
HŽ Putnički prijevoz d.o.o.               8.25
SJ AB                  

### Normalisation des variables numériques

Le StandardScaler centre chaque variable autour de 0 et la réduit 
à un écart-type de 1. Cela évite qu'une variable avec de grandes valeurs 
comme **distance_km** (389 à 1847) domine une variable avec de petites valeurs 
comme **type_service** (0 ou 1).

Le scaler est fitté uniquement sur le train puis appliqué 
sur la validation et le test, pour les mêmes raisons que le Target Encoding : 
éviter le data leakage.

In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)

# Reconvertir en DataFrame
X_train_scaled = pd.DataFrame(X_train_scaled, columns=features_regression)
X_val_scaled   = pd.DataFrame(X_val_scaled,   columns=features_regression)
X_test_scaled  = pd.DataFrame(X_test_scaled,  columns=features_regression)

print("Normalisation OK")
print(X_train_scaled.describe().round(2))

Normalisation OK
       distance_km  operateur  type_service  duree_trajet_min
count       275.00     275.00        275.00            275.00
mean         -0.00       0.00          0.00              0.00
std           1.00       1.00          1.00              1.00
min          -1.54      -2.64         -1.76             -1.91
25%          -0.78      -0.33          0.57             -0.80
50%          -0.15       0.21          0.57              0.06
75%           0.62       0.49          0.57              0.67
max           3.59       1.55          0.57              2.34


### Résultat de la normalisation

Après normalisation, toutes les variables ont une moyenne de 0 et un écart-type de 1.

Avant la normalisation, **distance_km** allait de 389 à 1847 
et **type_service** valait 0 ou 1. Le modèle aurait accordé plus d'importance 
à **distance_km** uniquement parce que ses chiffres sont plus grands, 
pas parce qu'elle est plus informative.

Après la normalisation, toutes les variables partent sur un pied d'égalité. 
Le modèle peut comparer leur influence de façon équitable.

**type_service** présente des percentiles identiques au 25e, 50e et 75e (0.57). 
C'est normal car c'est une variable binaire : le StandardScaler la transforme 
mais elle ne prend que deux valeurs distinctes.

### Conclusion du notebook de preprocessing

Ce notebook avait pour objectif d'explorer et valider chaque transformation 
appliquée au dataset brut. Travailler dans un notebook permet d'observer 
le résultat de chaque étape sur les données réelles et d'ajuster les décisions 
de transformation en conséquence.

Les étapes suivantes ont été validées :

La suppression des colonnes non pertinentes pour les modèles.
Le calcul de **duree_trajet_min** à partir des heures de départ et d'arrivée, 
avec détection et suppression de 7 anomalies.
L'encodage binaire de **type_service** et le Target Encoding de **operateur**, 
appliqués après le split pour éviter tout data leakage.
La normalisation via StandardScaler fitté uniquement sur le train.
Le split 70/15/15 avec random_state=42.

L'ensemble de ces transformations sera maintenant industrialisé 
dans `src/preprocessing.py`, le script final reproductible 
qui pourra être exécuté directement en ligne de commande.